# Imports & File Paths


In [1]:
import pandas as pd

# File paths
NEW_CONTEST_FILE = "sheet1_standings.csv"
MASTER_FILE = "test_standing.csv"

# Load Data & Initialize Master Sheet

In [3]:
# Load newly generated contest sheet
df_new = pd.read_csv(NEW_CONTEST_FILE)

# Prepare subset of data containing scores and fastest solves
update_cols = ["Handle", "Score"]
if "Fastest_Solve" in df_new.columns:
    update_cols.append("Fastest_Solve")

df_updates = df_new[update_cols].copy()

# Try loading existing master file
try:
    df_master = pd.read_csv(MASTER_FILE)
    print("Loaded existing Master Sheet.")
except FileNotFoundError:
    print("Master Sheet not found. Initializing new Master Sheet.")
    df_master = df_updates.copy()

    # Re-assign Ranks based on score
    df_master = df_master.sort_values(by="Score", ascending=False).reset_index(
        drop=True
    )
    df_master["Rank"] = df_master.index + 1
    df_master["Rank_Change"] = 0  # Initial run has 0 rank change

    df_master.to_excel(MASTER_FILE, index=False)

Loaded existing Master Sheet.


# Merge, Accumulate Scores & Calculate Rank Difference

In [4]:
# --- Pre-check: Ensure 'Handle' column exists in both DataFrames ---
for df in [df_master, df_updates]:
    if "Handle" not in df.columns:
        if df.index.name in ["Handle", "Codeforces Handle"]:
            df.reset_index(inplace=True)
        elif "Codeforces Handle" in df.columns:
            df.rename(columns={"Codeforces Handle": "Handle"}, inplace=True)
        elif "CF_Handle" in df.columns:
            df.rename(columns={"CF_Handle": "Handle"}, inplace=True)

if "Score" in df_master.columns and df_master is not df_updates:

    # 1. Save previous rank before merging
    if "Rank" in df_master.columns:
        old_ranks = df_master.set_index("Handle")["Rank"].to_dict()
    else:
        old_ranks = {
            handle: idx + 1 for idx, handle in enumerate(df_master["Handle"])
        }

    # 2. Merge master sheet with new contest updates
    merged = pd.merge(
        df_master,
        df_updates,
        on="Handle",
        how="outer",
        suffixes=("_master", "_new"),
    )

    # Fill NaNs with 0 for scores
    merged = merged.fillna(0)

    # 3. Accumulate Scores
    merged["Score"] = merged["Score_master"] + merged["Score_new"]

    # Accumulate Fastest_Solve if present
    if "Fastest_Solve_new" in merged.columns:
        if "Fastest_Solve_master" in merged.columns:
            merged["Fastest_Solve"] = (
                merged["Fastest_Solve_master"] + merged["Fastest_Solve_new"]
            )
        else:
            merged["Fastest_Solve"] = merged["Fastest_Solve_new"]
    elif "Fastest_Solve_master" in merged.columns:
        merged["Fastest_Solve"] = merged["Fastest_Solve_master"]

    # 4. Sort leaderboard by new total Score descending
    cols = ["Handle", "Score"]
    if "Fastest_Solve" in merged.columns:
        cols.append("Fastest_Solve")

    df_master = (
        merged[cols].sort_values(by="Score", ascending=False).reset_index(drop=True)
    )

    # 5. Assign New Rank
    df_master["Rank"] = df_master.index + 1

    # 6. Calculate Rank Change (Old Rank - New Rank)
    def calc_rank_change(row):
        handle = row["Handle"]
        new_rank = row["Rank"]

        if handle in old_ranks:
            old_rank = old_ranks[handle]
            diff = old_rank - new_rank
            if diff > 0:
                return f"+{diff}"
            return str(diff)
        else:
            return "NEW"  # First-time participant

    df_master["Rank_Change"] = df_master.apply(calc_rank_change, axis=1)

    # 7. Final column ordering
    final_cols = ["Rank", "Rank_Change", "Handle", "Score"]
    if "Fastest_Solve" in df_master.columns:
        final_cols.insert(4, "Fastest_Solve")

    df_master = df_master[final_cols]

# Export & Display Results

In [5]:
# Save updated Master Sheet
df_master.to_csv(MASTER_FILE, index=False)

print("Master Sheet updated successfully!")
df_master.head(10)
df_master

Master Sheet updated successfully!


,Rank,Rank_Change,Handle,Score,Fastest_Solve
0,1,NEW,HosSam_Abu_elez,650.0,0.0
1,2,NEW,Yousef_maaty20,520.0,1.0
2,3,NEW,Hend_22,480.0,4.0
3,4,+9,Fa7L,320.0,1.0
4,5,+41,_7azzzemm_,300.0,0.0
...,...,...,...,...,...
112,113,-58,tawfee_k226,0.0,0.0
113,114,-99,yasmineelgharib063,0.0,0.0
114,115,-67,yousef220,0.0,0.0
115,116,-42,zainabalii,0.0,0.0
